# Tahap 2 - Case Representation KDRT
Notebook ini hanya untuk Tahap 2 sesuai instruksi tugas.

In [1]:
import os,re,json,pandas as pd,numpy as np

## Load hasil Tahap 1

In [5]:
from google.colab import drive
import os

drive.mount('/content/drive')

RAW_FOLDER = '/content/drive/My Drive/TugasPK/Test/raw_pdf/data/raw'

if os.path.exists(RAW_FOLDER):
    files = sorted([f for f in os.listdir(RAW_FOLDER) if f.endswith('.txt')])
    print(f"Berhasil terhubung ke Drive! Ditemukan {len(files)} file .txt.")
else:
    print(f"Folder tidak ditemukan di path: {RAW_FOLDER}")
    print("Pastikan Google Drive sudah ter-mount dan path-nya benar.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Berhasil terhubung ke Drive! Ditemukan 30 file .txt.


## Metadata Extraction

In [6]:
def extract_nomor_perkara(text):
 m=re.search(r'(\d+\s*[A-Za-z./-]+\s*/\s*\d{4})',text,re.I)
 return m.group(1) if m else 'UNKNOWN'

def extract_tahun(text):
 m=re.search(r'(19\d{2}|20\d{2})',text)
 return m.group(1) if m else 'UNKNOWN'

def extract_pasal(text):
 x=re.findall(r'pasal\s+\d+[^\n\.]{0,120}',text,re.I)
 x=list(dict.fromkeys([i.strip() for i in x]))
 return '; '.join(x[:10]) if x else 'UNKNOWN'

def extract_pengadilan(text):
 m=re.search(r'pengadilan negeri\s+([a-z\s]+)',text,re.I)
 return m.group(1).strip() if m else 'UNKNOWN'

def extract_tingkat(text):
 t=text.lower()
 if 'kasasi' in t: return 'Kasasi'
 if 'banding' in t: return 'Banding'
 return 'Tingkat Pertama'


## Ekstraksi Pihak

In [7]:
def extract_terdakwa(text):
 m=re.search(r'terdakwa\s*[:\-]?\s*([a-z\s\.]+)',text,re.I)
 return m.group(1).strip() if m else 'UNKNOWN'

def extract_korban(text):
 pola=[r'korban\s*[:\-]?\s*([a-z\s\.]+)',r'istri\s+([a-z\s\.]+)',r'saksi korban\s*([a-z\s\.]+)']
 for p in pola:
  m=re.search(p,text,re.I)
  if m: return m.group(1).strip()
 return 'UNKNOWN'


## Key Content Extraction

In [8]:
def extract_ringkasan_fakta(text):
 idx=text.find('menimbang')
 bagian=text[:idx] if idx!=-1 else text
 return ' '.join(bagian.split()[:500])

def extract_argumen_hukum(text):
 s=text.find('menimbang'); e=text.find('mengadili')
 if s!=-1 and e!=-1 and e>s: return text[s:e]
 return 'UNKNOWN'

def extract_amar(text):
 i=text.find('mengadili')
 return text[i:i+2500] if i!=-1 else 'UNKNOWN'


## Jenis KDRT & Outcome

In [9]:
def classify_kdrt(text,pasal):
 t=(text+' '+pasal).lower()
 if 'pasal 49' in t or 'nafkah' in t: return 'Penelantaran'
 if 'pasal 45' in t: return 'Psikis'
 return 'Fisik'

def extract_outcome(amar):
 a=amar.lower()
 if 'pidana percobaan' in a: return 'pidana_percobaan'
 if 'pidana penjara' in a: return 'pidana_penjara'
 if 'membebaskan terdakwa' in a: return 'bebas'
 if 'lepas dari segala tuntutan hukum' in a: return 'lepas'
 if 'menolak permohonan kasasi' in a: return 'kasasi_ditolak'
 if 'mengabulkan permohonan kasasi' in a: return 'kasasi_diterima'
 return 'unknown'


## Feature Engineering

In [10]:
def build_retrieval_text(jenis,pasal,fakta,arg):
 return f'{jenis} {pasal} {fakta} {arg}'

def word_count(text): return len(text.split())
def unique_words(text): return len(set(text.split()))

## Build Dataset

In [11]:
records=[]
for idx,file in enumerate(files,start=1):
 text=open(os.path.join(RAW_FOLDER,file),encoding='utf-8').read()
 pasal=extract_pasal(text)
 fakta=extract_ringkasan_fakta(text)
 arg=extract_argumen_hukum(text)
 amar=extract_amar(text)
 jenis=classify_kdrt(text,pasal)
 records.append({
 'case_id':idx,'file_name':file,'nomor_perkara':extract_nomor_perkara(text),'tahun':extract_tahun(text),'tingkat_peradilan':extract_tingkat(text),'pengadilan_asal':extract_pengadilan(text),'jenis_kdrt':jenis,'pasal':pasal,'terdakwa':extract_terdakwa(text),'korban':extract_korban(text),'ringkasan_fakta':fakta,'argumen_hukum':arg,'amar_putusan':amar,'outcome':extract_outcome(amar),'retrieval_text':build_retrieval_text(jenis,pasal,fakta,arg),'word_count':word_count(text),'unique_words':unique_words(text)})
cases=pd.DataFrame(records)
cases.head()

,case_id,file_name,nomor_perkara,tahun,tingkat_peradilan,pengadilan_asal,jenis_kdrt,pasal,terdakwa,korban,ringkasan_fakta,argumen_hukum,amar_putusan,outcome,retrieval_text,word_count,unique_words
0,1,case_001.txt,UNKNOWN,2014,Kasasi,tanjungkara ng sejak tanggal h k,Fisik,pasal 44 ayat 1 undang undang nomor 23 tahun 2...,n a nama sandy ringgala s e bin i syamsul bahr...,duduk di a pinggir kasur dan anak saksi devpi ...,a i s e n o d n i k i l b u p e a r i s g e n ...,UNKNOWN,mengadili perkaranya dengan sengaja melakukan ...,unknown,Fisik pasal 44 ayat 1 undang undang nomor 23 t...,5719,816
1,2,case_002.txt,UNKNOWN,2023,Kasasi,UNKNOWN,Fisik,pasal 44 ayat 2 undang undang nomor 23 tahun 2...,i h nama triamana hadkianta a pangkat nrp sertu,mendapat jatuh sakit sebagaimana diatur dan s ...,a i s e n o d n i k i l b u p e a r i s g e n ...,UNKNOWN,UNKNOWN,unknown,Fisik pasal 44 ayat 2 undang undang nomor 23 t...,3895,551
2,3,case_003.txt,UNKNOWN,2010,Kasasi,mungkid karena n n didakwa buahwa terdakwa nur...,Penelantaran,pasal 34 ayat 1 undang undang e republik indon...,n nama nuryanto bin sunarto slamet i tempat la...,yang akhirnya e jumini mengandung dan melahirk...,a i s e n o d n i k i l b u p e a r i s g e n ...,UNKNOWN,UNKNOWN,unknown,Penelantaran pasal 34 ayat 1 undang undang e r...,3583,644
3,4,case_004.txt,UNKNOWN,2013,Kasasi,pekanbaru karena o u didakwa d g pertama n aba...,Fisik,pasal 44 ayat 4 undang unedang ri nomor 23 tah...,i n a m a said zainal bin said umar usman h k ...,wan maryam alias mar yang merupakan istri s m ...,a i s e n o d n i k i l b u p e a r i s g e n ...,menimbang bahwa putusan pengadilan tinggi ters...,mengadili tidak dilaksanakan menurut ketentuan...,kasasi_ditolak,Fisik pasal 44 ayat 4 undang unedang ri nomor ...,4610,693
4,5,case_005.txt,UNKNOWN,2018,Kasasi,padang o u karena didakwa dengan dakwaan tungg...,Penelantaran,pasal 49 huruf a undang n aundang nomor 23 tah...,telah memutus perkara terdakwa nama devia idru...,imerupakan suami h istri sesuai akta nikah nomor,a i s e n o d n i k i l b u p e a r i s g e n ...,menimbang bahwa putusan pengadilan tinggi pada...,mengadili tidak s m dilaksanakan gmenurut kete...,kasasi_ditolak,Penelantaran pasal 49 huruf a undang n aundang...,3210,528


## Quality Check

In [12]:
metadata_summary=pd.DataFrame({'jumlah_kasus':[len(cases)],'jumlah_pasal_unik':[cases.pasal.nunique()],'jumlah_outcome_unik':[cases.outcome.nunique()],'jumlah_jenis_kdrt':[cases.jenis_kdrt.nunique()],'rata_word_count':[cases.word_count.mean()]})
metadata_summary

,jumlah_kasus,jumlah_pasal_unik,jumlah_outcome_unik,jumlah_jenis_kdrt,rata_word_count
0,30,30,4,3,5391.2


## Export Output

In [14]:
PROCESSED_FOLDER = '/content/drive/My Drive/TugasPK/Test/raw_pdf/data/processed'
os.makedirs(PROCESSED_FOLDER, exist_ok=True)

cases.to_csv(os.path.join(PROCESSED_FOLDER, 'cases.csv'), index=False)
cases.to_json(os.path.join(PROCESSED_FOLDER, 'cases.json'), orient='records', force_ascii=False, indent=2)

metadata_summary.to_csv(os.path.join(PROCESSED_FOLDER, 'metadata_summary.csv'), index=False)
cases[['case_id','word_count','unique_words']].to_csv(os.path.join(PROCESSED_FOLDER, 'feature_summary.csv'), index=False)

print(f'Output Tahap 2 BERHASIL disimpan secara permanen di Drive: {PROCESSED_FOLDER}')

Output Tahap 2 BERHASIL disimpan secara permanen di Drive: /content/drive/My Drive/TugasPK/Test/raw_pdf/data/processed
